# Исследование обучения GAN на Fashion-MNIST

В этом ноутбуке кратко разобран код учебного мини-проекта. Здесь по блокам объясняется, что делает каждая часть проекта и какие результаты были получены после обучения модели.

## Структура проекта

Проект состоит из нескольких основных файлов:

- `config.py` — хранит параметры обучения и пути к папкам.
- `models.py` — содержит классы генератора и дискриминатора.
- `utils.py` — содержит вспомогательные функции для сохранения результатов и настройки воспроизводимости.
- `train_gan.py` — основной скрипт обучения GAN.
- `generate_samples.py` — скрипт для генерации новых изображений после обучения модели.

## Блок параметров проекта

В файле `config.py` задаются основные гиперпараметры: размер батча, размер скрытого вектора, скорость обучения, число эпох и частота сохранения изображений. Также здесь задаются пути, по которым сохраняются веса модели, графики и изображения.

In [ ]:
import config

print('batch_size =', config.batch_size)
print('latent_dim =', config.latent_dim)
print('learning_rate =', config.learning_rate)
print('epochs =', config.epochs)
print('sample_interval =', config.sample_interval)
print('random_seed =', config.random_seed)

## Архитектура генератора

Генератор получает случайный вектор размерности 100 и последовательно преобразует его через несколько полносвязных слоёв. Между слоями используются `BatchNorm` и `ReLU`, что помогает сделать обучение стабильнее. На выходе применяется функция `Tanh`, после чего вектор преобразуется в изображение размера `1x28x28`.

## Архитектура дискриминатора

Дискриминатор получает изображение, разворачивает его в одномерный вектор и пропускает через несколько полносвязных слоёв. Для нелинейности используется `LeakyReLU`, а финальный слой с `Sigmoid` выдаёт вероятность того, что изображение является реальным.

In [ ]:
from models import Generator, Discriminator

generator = Generator()
discriminator = Discriminator()

print(generator)
print()
print(discriminator)

## Вспомогательные функции

В файле `utils.py` находятся вспомогательные функции. Они отвечают за фиксацию случайного зерна, создание папок, инициализацию весов, сохранение изображений, построение графика потерь, а также за сохранение истории обучения в `CSV` и итоговой сводки в `JSON`.

## Подготовка данных

В `train_gan.py` есть функция `get_dataloader()`. Она загружает датасет Fashion-MNIST, преобразует изображения в тензоры и нормализует их в диапазон `[-1, 1]`. Это важно, потому что генератор использует `Tanh` на выходе и тоже работает в этом диапазоне.

## Инициализация обучения

Перед началом обучения фиксируется `seed`, создаются нужные папки, определяется устройство выполнения (`CPU` или `CUDA`), инициализируются генератор и дискриминатор, выбирается функция потерь `BCELoss` и оптимизаторы `Adam`.

## Обучение дискриминатора

На каждом батче дискриминатор сначала получает реальные изображения с метками `1`, затем сгенерированные изображения с метками `0`. После этого вычисляется ошибка на обеих частях и обновляются веса дискриминатора.

## Обучение генератора

После шага дискриминатора обучается генератор. Он создаёт изображения из случайного шума, затем эти изображения подаются в дискриминатор. Генератор обучается так, чтобы дискриминатор принимал его изображения за реальные, поэтому для него используются целевые метки `1`.

## Сохранение результатов

После каждой нужной эпохи проект сохраняет сетку изображений генератора. После завершения обучения сохраняются веса модели, график потерь, итоговая сетка изображений, история обучения по эпохам и JSON-сводка с результатами запуска.

## Генерация изображений после обучения

Файл `generate_samples.py` нужен для отдельной генерации новых изображений. Он загружает сохранённые веса генератора, создаёт случайный шум и сохраняет новую сетку изображений без повторного обучения модели.

In [ ]:
import json
import pandas as pd

history = pd.read_csv('outputs/training_history.csv')
history.head()

In [ ]:
with open('outputs/training_summary.json', 'r', encoding='utf-8') as file:
    summary = json.load(file)

summary

## Визуальные результаты

Ниже можно вывести график потерь и общую картинку прогресса генерации. По ним видно, что в начале генератор создаёт шум, затем появляются контуры, а к последним эпохам изображения становятся похожими на реальные объекты одежды.

In [ ]:
from IPython.display import Image, display

display(Image('outputs/plots/loss_curve.png'))
display(Image('outputs/plots/training_progress.png'))

## Вывод

В ходе проекта была реализована простая GAN для Fashion-MNIST. По сохранённым изображениям видно, что модель действительно научилась переходить от генерации шума к более осмысленным изображениям одежды. Это подтверждает, что обучение прошло корректно и проект можно использовать как полноценную учебную работу.